In [ ]:
def write_runscript_from_config(config_data: dict, output_path: str):
    """
    根据配置字典生成一个类似 runscript_test 的固定格式运行文件。
    该版本能处理值为列表（每场景不同）或字符串（所有场景通用）的参数。

    参数:
    config_data (dict): 包含所有模拟参数的字典。
    output_path (str): 输出文件的路径。
    """
    lines = []
    
    # --- 1. 提取通用设置 ---
    general_config = config_data.get("SETUP_GENERAL", {})
    
    grid_line = ' '.join(map(str, general_config.get("grid_dims", [1, 1, 1, 1, 1])))
    lines.append(grid_line)
    
    lines.append(general_config.get("site_data_file", "st022852"))
    lines.append(general_config.get("topography_data_file", "tp022852"))
    
    num_scenes = general_config.get("num_scenes", 0)
    num_runs = general_config.get("num_runs", 1)
    lines.append(f"{num_scenes} {num_runs}")
    
    # --- 2. 循环处理每个场景 ---
    scenes_config = config_data.get("SETUP_SCENES", {})
    # 定义场景中文件参数的顺序
    scene_file_keys = [
        "weather_data_files", "weather_options_files", "land_management_files",
        "plant_management_files", "soil_output_1", "atmospheric_output",
        "n_flux_output", "p_flux_output", "soil_output_2", "soil_output_3",
        "water_props_output", "n_props_output", "p_props_output", "t_props_output"
    ]

    for i in range(num_scenes):
        lines.append("1 1")
        
        for key in scene_file_keys:
            config_value = scenes_config.get(key)
            
            # **智能处理逻辑**
            # 如果值是列表，则按场景索引取值
            if isinstance(config_value, list):
                if i < len(config_value):
                    lines.append(config_value[i])
                else:
                    lines.append("NO_FILE_SPECIFIED_IN_LIST")
            # 如果值是字符串，则所有场景都使用该值
            elif isinstance(config_value, str):
                lines.append(config_value)
            # 如果未提供值
            else:
                lines.append("NO_FILE_SPECIFIED")

    # --- 3. 添加结束标志 ---
    lines.append("0 0")

    # --- 4. 将所有行写入文件 ---
    try:
        with open(output_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"文件已成功生成在: {output_path}")
    except IOError as e:
        print(f"写入文件时出错: {e}")


# =======================================================================
#                           --- 使用示例 ---
# =======================================================================

if __name__ == '__main__':
    # 1. 定义新的配置字典，注意值的变化
    my_config = {
        "SETUP_GENERAL": {
            "grid_dims": [1, 1, 1, 1, 1],
            "site_data_file": "st022852",
            "topography_data_file": "tp022852",
            "num_scenes": 2,
            "num_runs": 1
        },
        "SETUP_SCENES": {
            # 这个参数保持为列表，因为每个场景的气象数据通常不同
            "weather_data_files":     ['w1980022852', 'w1981022852'], 
            # --- 以下参数已从列表改为单个字符串 ---
            "weather_options_files":  ['opt1800', 'opt1801'],
            "land_management_files":  'sm',
            "plant_management_files": ['pft_arctic_p', 'pft_arctic_g'],
            "soil_output_1":          'NO',
            "atmospheric_output":     'NO',
            "n_flux_output":          'NO',
            "p_flux_output":          'NO',
            "soil_output_2":          'NO',
            "soil_output_3":          'dc',
            "water_props_output":     'dw',
            "n_props_output":         'dn',
            "p_props_output":         'NO',
            "t_props_output":         'dh'
        }
    }

    # 2. 调用函数，指定输出文件名
    output_filename = "runscript_generated_single_element.txt"
    write_runscript_from_config(my_config, output_filename)